In [270]:
import ollama
import json

In [271]:
with open("data/grids.json", "r") as f:
    grids = json.load(f)

In [272]:
MODEL = "mistral"

In [274]:
SYSTEM_PROMPT = """
You are a helpful assistant that helps a robot navigate a grid world. 

A robot is placed in a 5x5 grid world:
    - The top left corner of the grid is (0, 0) and the bottom right corner is (4, 4).
    - 'S' represents the starting position of the robot.
    - 'G' represents the goal position. 
    - '.' represents an open cell.
    - 'B' represents a blocked cell.
    - The robot should find the shortest path from the cell 'S' to the cell 'G'.
    - The robot can only move to open cells ('.').
    - The robot cannot move into blocked cells ('B').
    - The robot can move to four directions: North, South, East, and West.
    - North moves the robot up by one cell. The robot will move from (x, y) to (x-1, y).
    - South moves the robot down by one cell. The robot will move from (x, y) to (x+1, y).
    - East moves the robot right by one cell. The robot will move from (x, y) to (x, y+1).
    - West moves the robot left by one cell. The robot will move from (x, y) to (x, y-1).
    - The robot cannot move outside the boundaries of the grid.
    - Do not provide any explanations or additional text. Only provide the array of directions as the output.

Respond with an array of directions (North, South, East, West) that the robot should take to reach the goal from the starting position. 

#####################

Example 1:

Input:
Grid:
S B . B .
. . B B .
. . . . .
. . . . .
. . . . G

Starting position: [0, 0]
Goal position: [4, 4]

Output:
['South', 'East', 'South', 'East', 'East', 'East', 'South', 'South']

#####################

Example 2:

Input:
Grid:
. . . . .
. G B . .
. . B B S
. . B B .
. . . B .

Starting position: [2, 4]
Goal position: [1, 1]

Output:
['North', 'West', 'North', 'West', 'West', 'South']

"""

In [275]:
def ask_slm(prompt):
    print("Asking SLM...")
    response = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        options={
            "temperature": 0.0, 
            "seed": 42
        }
    )

    return response["message"]["content"]

In [277]:
for grid_info in grids[:5]:
    grid = grid_info["grid"]
    start = grid_info["start"]
    goal = grid_info["goal"]

    prompt = f"Grid:\n{'\n'.join(' '.join(row) for row in grid_info['grid'])}\n\nStarting position: {start}\nGoal position: {goal}"
    print(f"Prompt:\n{prompt}\n")

    response = ask_slm(prompt)
    print(f"Response:\n{response}\n")


Prompt:
Grid:
S . . . B
B . . . .
. G . . .
. . . . B
. . . B .

Starting position: [0, 0]
Goal position: [2, 1]

Asking SLM...
Response:
 ['South', 'East', 'South', 'East', 'South', 'East']

Prompt:
Grid:
. . B . S
. . B B .
. . . . .
G . . B .
. . . . .

Starting position: [0, 4]
Goal position: [3, 0]

Asking SLM...
Response:
 ['South', 'West', 'South', 'West', 'West', 'South', 'West', 'South', 'East']

Prompt:
Grid:
. . . . .
. . B B .
. . . B B
. . B . .
S G B . .

Starting position: [4, 0]
Goal position: [4, 1]

Asking SLM...
Response:
 ['South']

Prompt:
Grid:
. B . B .
. . . B .
. . G B .
S B . . B
. . . . .

Starting position: [3, 0]
Goal position: [2, 2]

Asking SLM...
Response:
 ['South', 'West', 'South', 'West', 'West', 'North', 'East']

Prompt:
Grid:
. B B . S
. . B . .
. . . . G
. . . . .
B . . . B

Starting position: [0, 4]
Goal position: [2, 4]

Asking SLM...
Response:
 ['South', 'East', 'South', 'East', 'South']



In [260]:
user_prompt = f"Grid:\n{'\n'.join(' '.join(row) for row in grid_info['grid'])}\nStarting position: {grid_info['start']}\nGoal position: {grid_info['goal']}."

In [261]:
print(user_prompt)

Grid:
. . . B B
B . . . S
. . . . .
G . . . .
. . B . .
Starting position: [1, 4]
Goal position: [3, 0].


In [262]:
ask_slm(user_prompt)

Asking SLM...


" To find the shortest path for the robot, we can use Breadth-First Search (BFS) algorithm. This algorithm explores all the nodes in the graph (in this case, the grid) in a breadth-first manner, i.e., it explores all the nodes at the current depth before moving on to nodes at the next depth level.\n\nHere's the BFS implementation for the given problem:\n\n```python\ndef bfs(grid, start, goal):\n    directions = [(0, 1), (1, 0), (0, -1), (-1, 0)]  # North, East, South, West\n    visited = set()\n    queue = [start]\n\n    while queue:\n        pos = queue.pop(0)\n        if pos == goal:\n            return _path(pos, start, directions)\n\n        visited.add(pos)\n        for dx, dy in directions:\n            next_pos = (pos[0] + dx, pos[1] + dy)\n            if 0 <= next_pos[0] < len(grid) and 0 <= next_pos[1] < len(grid[0]) and grid[next_pos[0]][next_pos[1]] != 'B' and next_pos not in visited:\n                queue.append(next_pos)\n\n    return None\n\ndef _path(end, start, directi

In [ ]:
SYSTEM_PROMPT = """
You are a grid world navigation engine. 

A robot is on a 5x5 coordinate plane where (0,0) is top-left and (4,4) is bottom-right.
Movement rules:
- North: (x-1, y)
- South: (x+1, y)
- East:  (x, y+1)
- West:  (x, y-1)

CRITICAL: Do not output any Python code, BFS algorithms, or introductory conversational text. You must output ONLY a valid JSON array of strings representing the path directions.

Input Format:
Starting position: [x, y]
Goal position: [x, y]
Blocked cells: [[x, y], [x, y]...]
Open cells: All cells not blocked.
"""